In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from tqdm import tqdm

# Configuração de dispositivo (usa GPU se disponível, senão CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
## Texto Bruto:  "O governo aprovou a lei"
##    ⬇
## Tokens:       ["o", "governo", "aprovou", "a", "lei"]
##    ⬇
## IDs (Vocab):  [10,  452,      33,       5,   99 ]
##    ⬇
## Padding:      [10,  452,      33,       5,   99,  0,   0,   0 ... ] (até chegar em 100)
##    ⬇
## SAÍDA X:      torch.tensor([10, 452, ...], dtype=torch.long)

In [4]:
train = pd.read_csv('data/train.csv', sep=';')  # Supondo que o arquivo CSV tenha colunas 'text' e 'label'
test = pd.read_csv('data/test.csv', sep=';')    # Supondo que o arquivo CSV tenha colunas 'text' e 'label'

In [5]:
train['content'] = train['title'] + '/n' + train['text']
test['content'] = test['title'] + '/n' + test['text']

In [6]:
train['len'] = train['content'].apply(lambda x: len(x.split(' ')))

In [7]:
train['len'].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])

count    24353.000000
mean       431.571264
std        354.728471
min          1.000000
25%        229.000000
50%        388.000000
75%        541.000000
90%        778.000000
95%        936.000000
99%       1516.400000
max       8122.000000
Name: len, dtype: float64

In [8]:
### Criando "vocabulário" simples
from collections import Counter

def build_vocab(texts, max_size=10000):
    """
    Lê todas as notícias e cria um dicionário das palavras mais comuns.
    """
    # 1. Conta todas as palavras de todas as notícias
    word_counter = Counter()
    for text in texts:
        
        tokens = str(text).lower().split() 
        word_counter.update(tokens)
    
    # 2. Pega as 'max_size' palavras mais comuns
    most_common = word_counter.most_common(max_size)
    
    # 3. Cria o dicionário {palavra: id}
    # Começamos o ID em 1, porque o 0 é reservado para Padding/Desconhecido
    vocab = {word: idx + 1 for idx, (word, count) in enumerate(most_common)}
    
    return vocab

In [9]:
vocab = build_vocab(train['content'].to_list(), max_size=10000)
vocab

{'the': 1,
 'to': 2,
 'of': 3,
 'a': 4,
 'and': 5,
 'in': 6,
 'that': 7,
 'on': 8,
 'for': 9,
 'is': 10,
 's': 11,
 'he': 12,
 'with': 13,
 'trump': 14,
 'it': 15,
 'was': 16,
 'as': 17,
 'said': 18,
 'his': 19,
 'by': 20,
 'has': 21,
 'be': 22,
 'have': 23,
 'from': 24,
 'not': 25,
 'at': 26,
 'are': 27,
 'this': 28,
 'who': 29,
 'an': 30,
 'they': 31,
 'but': 32,
 'would': 33,
 'i': 34,
 'we': 35,
 'will': 36,
 'u.s.': 37,
 'about': 38,
 'their': 39,
 'had': 40,
 'president': 41,
 'you': 42,
 'been': 43,
 'were': 44,
 'or': 45,
 'after': 46,
 't': 47,
 'which': 48,
 'more': 49,
 'people': 50,
 'she': 51,
 'if': 52,
 '-': 53,
 'one': 54,
 'its': 55,
 'her': 56,
 'all': 57,
 'what': 58,
 'out': 59,
 'also': 60,
 'new': 61,
 'when': 62,
 'over': 63,
 'donald': 64,
 'up': 65,
 'state': 66,
 'no': 67,
 'said.': 68,
 'can': 69,
 'there': 70,
 'than': 71,
 'republican': 72,
 'just': 73,
 'our': 74,
 'house': 75,
 'could': 76,
 'some': 77,
 'other': 78,
 'united': 79,
 'government': 80,
 'in

## Vocabulário → Dataset → DataLoader → Modelo → Treino.

In [10]:
# ==========================================
# 1. PREPARAÇÃO DOS DADOS (Dataset)
# ==========================================

# Aqui precisamos pensar em uma noticia apenas.
# Não vamos preparar todas.

class FakeNewsDataset(Dataset):
    def __init__(self, content, labels, vocab, max_len=780):

        self.content = content
        self.labels = labels
        self.vocab = vocab # Dicionário {'palavra': id}
        self.max_len = max_len

    def __len__(self):
        return len(self.content)

    def __getitem__(self, idx):

        # Pega o dado original e cria novas variáveis para modelarmos....
        text = self.content[idx]
        label = self.labels[idx]
        
        # A. Tokenização simples (em caso real, use bibliotecas)
        # Separa para ex: ["o", "governo", "aprovou", "a", "lei"]
        tokens = text.lower().split()
        
        # B. Conversão para IDs (Numericalization)
        # Se a palavra não existe, usa 0 (padding/unknown)
        # Usa o vocabulário para converter cada token em ID
        ids = [self.vocab.get(token, 0) for token in tokens]
        
        # C. Padding/Truncating (Garantir tamanho fixo)
        # Todas entredas devem ter o mesmo tamanho do max_len logo:
        if len(ids) < self.max_len:
            ids = ids + [0] * (self.max_len - len(ids)) # Preenche com zeros
        else:
            ids = ids[:self.max_len] # Corta o excesso
            
        # D. Retorno como Tensor
        # IDs devem ser LongTensor (inteiros)
        # Label deve ser FloatTensor (para BCELoss)
        return torch.tensor(ids, dtype=torch.long), torch.tensor(label, dtype=torch.float)

"A camada oculta consegue resumir a frase porque ela é recorrente. Em cada passo, a LSTM recebe não apenas a palavra atual, mas também o 'Hidden State' do passo anterior. Isso permite que ela acumule contexto ao longo do tempo. O vetor final de 128 dimensões que enviamos para a camada Linear não representa apenas a última palavra, mas sim o estado final da memória da rede após processar a sequência completa."

In [11]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128):
        # Herdaremos todas as funcionalidades do nn.Module
        super().__init__()
        # Aqui definimos vocab_size+1 para incluir o padding 0
        # Embed_dim é o tamanho do vetor estamos embed_dim=100, você está descrevendo cada palavra com 100 características diferentes.
        # (linhas = vocab_size+1, colunas = embed_dim) matriz  de embedding
        # Ele vai treinar o embedding junto com o modelo
        self.embedding = nn.Embedding(vocab_size + 1, embed_dim)
        # Como ele vai ler cada palavra? precisa ter a dimensão de embed_dim
        # hidden_dim = Define quantos números a rede pode usar para descrever o "resumo" da frase que ela leu até agora.
        # A rede lê as palavras uma a uma, atualizando seu estado oculto. A quantidade de palavras só importa no DataLoad, aqui ele vai processar o que vier.
        # primeira dimensão é o lote ou seja primeira linha é meu bach se não ele lê coluna
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        #Retornar qualquer número real
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        # Pegamos a saída do último estado oculto e jogamos no .fc
        return self.fc(hidden[-1]) # Saída do último estado oculto

In [12]:
# 1. Definimos o tamanho do vocabulário baseado no dicionário que criamos
vocab_size = len(vocab) 

# 2. Criamos o objeto do modelo
# Passamos o vocab_size, e os outros (embed_dim e hidden_dim) já têm valores padrão (100 e 128)
model = LSTMModel(vocab_size=vocab_size).to(device)

print(model) # Isso vai imprimir a estrutura da sua rede para conferência

LSTMModel(
  (embedding): Embedding(10001, 100)
  (lstm): LSTM(100, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)


In [ ]:
# 1. O Juiz (Loss Function)
# BCEWithLogitsLoss: Especial para classificação 0 ou 1. 
# Ela aplica a Sigmoid internamente para você.
criterion = nn.BCEWithLogitsLoss()

# 2. O Professor (Optimizer)
# Adam: Ajusta os pesos de forma inteligente (nem muito rápido, nem muito devagar).
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [16]:
# 1. Instanciamos o seu Dataset (a cozinha)
train_dataset = FakeNewsDataset(content=train['content'], labels=train['label'], vocab=vocab)

train_loader = DataLoader(
    dataset=train_dataset, 
    batch_size=32,      # Baixamos para testar
    shuffle=True, 
    num_workers=0       # 0 desativa o multiprocessamento e corre tudo na thread principal
)

In [17]:
train_loader

In [19]:
epochs = 5
model.to(device) # Garante que o modelo está na GPU

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    # tqdm aqui para vermos a barra de progresso
    for batch_idx, (ids, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}")):
        
        # 1. MOVER PARA GPU (Com atribuição!)
        ids = ids.to(device)
        labels = labels.to(device)
        
        # --- DIAGNÓSTICO (Opcional, só para conferir uma vez) ---
        if batch_idx == 0 and epoch == 0:
            print(f"\n[CHECK] Modelo: {next(model.parameters()).device} | Dados: {ids.device}")
        
        # 2. TREINO
        optimizer.zero_grad()
        outputs = model(ids)
        
        # 3. LOSS (Squeeze(1) para evitar erro em lotes pequenos)
        loss = criterion(outputs.squeeze(1), labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    print(f"Época {epoch+1} Finalizada | Loss: {total_loss/len(train_loader):.4f}")

Epoch 1:   0%|          | 1/762 [00:00<01:38,  7.71it/s]


[CHECK] Modelo: cuda:0 | Dados: cuda:0


Epoch 1: 100%|██████████| 762/762 [00:08<00:00, 86.30it/s]


Época 1 Finalizada | Loss: 0.5855


Epoch 2: 100%|██████████| 762/762 [00:08<00:00, 89.56it/s]


...

Época 3 Finalizada | Loss: 0.1843


Epoch 4: 100%|██████████| 762/762 [00:08<00:00, 90.28it/s]


Época 4 Finalizada | Loss: 0.1041


Epoch 5: 100%|██████████| 762/762 [00:08<00:00, 88.72it/s]

Época 5 Finalizada | Loss: 0.0741


In [20]:
# 1. Instanciamos o seu Dataset (a cozinha)
test_dataset = FakeNewsDataset(content=test['content'], labels=test['label'], vocab=vocab)

test_loader = DataLoader(
    dataset=test_dataset, 
    batch_size=32,      # Baixamos para testar
    shuffle=True, 
    num_workers=0       # 0 desativa o multiprocessamento e corre tudo na thread principal
)

In [21]:
from sklearn.metrics import f1_score

model.eval() # Modo de avaliação (desliga o aprendizado)
todas_preds = []
todos_labels = []

with torch.no_grad(): # Desliga o cálculo de gradientes (mais rápido e gasta menos memória)
    for ids, labels in test_loader:
        ids = ids.to(device)
        
        outputs = model(ids)
        # Transforma Logit em Probabilidade (0-1) e depois em Classe (0 ou 1)
        preds = (torch.sigmoid(outputs.squeeze()) > 0.5).int()
        
        todas_preds.extend(preds.cpu().numpy())
        todos_labels.extend(labels.numpy())

f1 = f1_score(todos_labels, todas_preds)
print(f"F1-Score no Teste: {f1:.4f}")

F1-Score no Teste: 0.9793
